# Notebook Description

This notebook performs the following data cleaning steps for `psgc_tables`:

1. Cleans and standardizes city and province names using PSGC reference tables.
2. Removes unnecessary columns and renames key columns for clarity.
3. Assigns special province codes for Metro Manila cities and Isabela (Basilan).
4. Combines city and province tables to create a unified geography dimension table.
5. Writes the cleaned geography dimension table to the silver folder.

File name: geography_dim

Output Columns: 
- cityCode: string
- cityName: string
- provinceCode: string
- provinceName: string
- islandGroupCode: string

In [0]:
from pyspark.sql import functions as F
import re
bronze_folder = 'abfss://bronze@asterisktotle01.dfs.core.windows.net/PHHousing'
city_df = spark.read.format('delta').load(bronze_folder + '/psgc_cities')

# Remove the city of and drop columns of oldName,psgc10DigitCode, districtCode
dim_city_clean = (
    city_df.
    withColumn("cityName",
               F.regexp_replace(
                   F.col("name"),
                   r"(?i)^(City of|Science City of|Island Garden City of)\s+",
            ""
               )
            ).withColumnRenamed("code","cityCode")
             .withColumnRenamed("name", "cityNameRaw")
             .drop("oldName")
             .drop("psgc10DigitCode")
             .drop("districtCode")
             .drop("cityNameClean")
    )

# Remove city on the last word (case insensitive)
dim_city_clean = dim_city_clean.withColumn(
        "cityName",
        F.regexp_replace(F.col("cityName"), r"(?i)\s+city$", "")
    )

# lower case all the word and drop the cityNameRaw
dim_city_clean = (dim_city_clean.withColumn("cityName", F.lower(F.col("cityName")))
                  .drop("cityNameRaw")
                )

# assign metro manila code "133900000" to the following locations
# assign isabela to basilan province code "150700000"
metro_manila_province = [ 
        "manila", "mandaluyong", "marikina", "pasig", "quezon", 
    "san juan", "caloocan", "malabon", "navotas", "valenzuela", 
    "las piñas", "makati", "muntinlupa", "parañaque", "pasay", 
    "pateros", "taguig"        
]
dim_city_clean = dim_city_clean.withColumn(
    "provinceCode",
    F.when(F.col("cityName").isin(metro_manila_province),
    F.lit("133900000"))
    .when( F.col("cityName") == 'isabela', F.lit("150700000"))
    .otherwise(F.col("provinceCode"))
)




In [0]:

province_df = spark.read.format('delta').load(bronze_folder + '/psgc_provinces')

#remove psgc10DigitCode and rename code to provinceCode
dim_province_clean = (
    province_df
    .withColumnRenamed("code", "provinceCode")
    .withColumnRenamed("name", "provinceName")
    .drop("psgc10DigitCode")

) 

# Build small dataframe for Metro Manila "ncr"
ncr_row = spark.createDataFrame(
    [("133900000", "Metro Manila", "040000000", "luzon")],
    ["provinceCode", "provinceName", "regionCode", "islandGroupCode"]
)
dim_province_clean = dim_province_clean.unionByName(ncr_row, True)

# Lower case the province name
dim_province_clean = dim_province_clean.withColumn("provinceName", F.lower(F.col("provinceName")))



In [0]:

# Combine city_clean and province_clean using full outer join
geography_dim = (
    dim_city_clean.alias("c")
    .join(
        dim_province_clean.alias("p"),
        F.col("c.provinceCode") == F.col("p.provinceCode"),
        "fullouter"
    ).select(
        F.col("c.cityCode"),
        F.col("c.cityName"),
        F.col("p.provinceCode"),
        F.col("p.provinceName"),
        F.col("p.islandGroupCode")
    )
)

dim_geography_city = (
    geography_dim
    .select("cityCode", "cityName", "provinceCode", "provinceName", "islandGroupCode")
    .dropDuplicates(["cityCode"])
    .withColumnRenamed("cityCode", "geographyFk")
    .withColumnRenamed("cityName", "geographyName")
    .withColumn("geographyLevel", F.lit("City"))
)

dim_geography_province = (
    geography_dim
    .select("provinceCode", "provinceName", "islandGroupCode")
    .dropDuplicates(["provinceCode"])
    .withColumnRenamed("provinceCode", "geographyFk")
    .withColumnRenamed("provinceName", "geographyName")
    .withColumn("geographyLevel", F.lit("Province"))
)

dim_geography_unified = dim_geography_city.unionByName(
    dim_geography_province, allowMissingColumns=True
)
silver_folder = 'abfss://silver@asterisktotle01.dfs.core.windows.net/PHHousing'
dim_geography_unified.write.format('delta').mode('overwrite').save(silver_folder + '/dim_geography')
print('Table named dim_geography is written to silver folder')
